Imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import ADASYN
from collections import Counter

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


Carregamento e Limpeza dos dados

In [3]:
nome_arquivo = '../dataset_classificacao_vogal_A_normal.csv'

try:
    df = pd.read_csv(nome_arquivo)
    print(f"📊 Dataset Carregado: {df.shape[0]} linhas, {df.shape[1]} colunas.")
except FileNotFoundError:
    print(f"Erro: Não encontrei o arquivo em '{nome_arquivo}'.")
    print("Verifique se o notebook está dentro de uma subpasta e o CSV na pasta principal.")

# Remover colunas inúteis
colunas_inuteis = ['Nome_Arquivo', 'Patologia_Original', 'ID_Gravacao', 'ID_Paciente']
df_limpo = df.drop(columns=colunas_inuteis, errors='ignore')
df_limpo = df_limpo.dropna()

# Idade (Arredondar)
if 'Idade' in df_limpo.columns:
    df_limpo['Idade'] = df_limpo['Idade'].round().astype(int)

# Sexo (Encoding)
le = LabelEncoder()
if 'Sexo' in df_limpo.columns:
    df_limpo['Sexo'] = le.fit_transform(df_limpo['Sexo'])

print("✅ Dados limpos e tratados.")

📊 Dataset Carregado: 1464 linhas, 16 colunas.
✅ Dados limpos e tratados.


Divisão Treino e Teste

In [4]:
# Separar X e y
X = df_limpo.drop('Grupo_Alvo', axis=1)
y = df_limpo['Grupo_Alvo']

# Dividir 80% Treino / 20% Teste
# stratify=y garante que a proporção de Câncer seja mantida igual nos dois grupos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"🔹 Treino Original: {X_train.shape[0]} amostras")
print(f"🔸 Teste (Intocado):  {X_test.shape[0]} amostras")

🔹 Treino Original: 1170 amostras
🔸 Teste (Intocado):  293 amostras


Scaler

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

print("✅ Dados normalizados (Média 0, Desvio Padrão 1).")
display(X_train_final.head(3))

✅ Dados normalizados (Média 0, Desvio Padrão 1).


,Idade,Sexo,f0_mean,f0_std,f0_min,f0_max,jitter_local,jitter_rap,shimmer_local,shimmer_db,hnr
0,0.808819,1.106497,0.487356,-0.257337,0.612354,0.352827,0.370862,0.420052,0.599291,0.571302,-0.917575
1,-1.076277,-0.903753,0.602598,-0.302590,0.703274,0.429742,-0.439149,-0.407126,-0.673611,-0.670217,0.326474
2,0.861183,-0.903753,0.517771,-0.211258,0.642124,0.399921,-0.509813,-0.510849,-0.518711,-0.525938,1.146502


Balanceamento ADASYN

In [ ]:
print("DISTRIBUIÇÃO ANTES (Treino):")
print(Counter(y_train))

try:
    print("\nTentando aplicar ADASYN...")
    adasyn = ADASYN(random_state=42)
    X_train_bal, y_train_bal = adasyn.fit_resample(X_train_final, y_train)
    
    sucesso_adasyn = True
    print("\n✅ SUCESSO! ADASYN aplicado.")
    print("NOVA DISTRIBUIÇÃO:")
    print(Counter(y_train_bal))
    
    # Visualização
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.countplot(x=y_train, palette='viridis')
    plt.title("Original")
    plt.subplot(1, 2, 2)
    sns.countplot(x=y_train_bal, palette='viridis')
    plt.title("Pós-ADASYN")
    plt.show()

except ValueError as e:
    print("\n" + "="*60)
    print("❌ FALHA NO ADASYN")
    print("="*60)
    print(f"Erro: {e}")
    print("Nenhum arquivo será salvo para evitar contaminação.")

DISTRIBUIÇÃO ANTES (Treino):
Counter({1: 567, 0: 549, 2: 54})

Tentando aplicar ADASYN...

❌ FALHA NO ADASYN
Erro: No samples will be generated with the provided ratio settings.
Nenhum arquivo será salvo para evitar contaminação.
